In [0]:
  %tensorflow_version 2.x

import os
import tensorflow as tf
import numpy as np
from prepare_data2 import parse_seq
import pickle

In [0]:
!python3 prepare_data2.py shakespeare_input.txt shake \\n\\n+ -m 500

Split input into 14279 sequences...
Longest sequence is 3082 characters. If this seems unreasonable, consider using the maxlen argument!
Removing sequences longer than 500 characters...
13575 sequences remaining.
Longest remaining sequence has length 499.
Removing length-0 sequences...
13575 sequences remaining.
Serialized 100 sequences...
Serialized 200 sequences...
Serialized 300 sequences...
Serialized 400 sequences...
Serialized 500 sequences...
Serialized 600 sequences...
Serialized 700 sequences...
Serialized 800 sequences...
Serialized 900 sequences...
Serialized 1000 sequences...
Serialized 1100 sequences...
Serialized 1200 sequences...
Serialized 1300 sequences...
Serialized 1400 sequences...
Serialized 1500 sequences...
Serialized 1600 sequences...
Serialized 1700 sequences...
Serialized 1800 sequences...
Serialized 1900 sequences...
Serialized 2000 sequences...
Serialized 2100 sequences...
Serialized 2200 sequences...
Serialized 2300 sequences...
Serialized 2400 sequences...

In [0]:
bs = 128
seq_len = 500
data = tf.data.TFRecordDataset("shake.tfrecords")
data = data.map(lambda x: parse_seq(x))
data = data.shuffle(46000).padded_batch(128, seq_len, drop_remainder=True).repeat()

vocab = pickle.load(open("shake_vocab", mode="rb"))
vocab_size = len(vocab)
ind_to_ch = {ind: ch for (ch, ind) in vocab.items()}

print(vocab_size)
print(vocab)

70
{'j': 3, 'Z': 4, 'a': 5, 'e': 6, 'R': 7, 'i': 8, 'n': 9, 'p': 10, 'l': 11, '-': 12, '$': 13, 'U': 14, 's': 15, 'M': 16, 'I': 17, '&': 18, 'E': 19, 't': 20, 'L': 21, 'X': 22, 'K': 23, 'S': 24, '!': 25, 'Y': 26, '\n': 27, 'r': 28, '?': 29, "'": 30, ']': 31, 'b': 32, '[': 33, 'k': 34, 'J': 35, '3': 36, 'f': 37, 'z': 38, 'O': 39, 'h': 40, 'x': 41, 'N': 42, 'A': 43, 'D': 44, 'V': 45, '.': 46, 'G': 47, 'u': 48, 'P': 49, 'C': 50, 'v': 51, 'B': 52, 'q': 53, 'm': 54, 'W': 55, 'Q': 56, 'g': 57, 'o': 58, 'F': 59, 'w': 60, 'd': 61, ',': 62, 'c': 63, ';': 64, ' ': 65, ':': 66, 'T': 67, 'H': 68, 'y': 69, '<PAD>': 0, '<S>': 1, '</S>': 2}


In [0]:
n_h = 512

layers = [tf.keras.Input(shape=(128,vocab_size), batch_size=128),
          tf.keras.layers.SimpleRNN(n_h, return_sequences=True, stateful=True),
          tf.keras.layers.Dense(vocab_size)]

model = tf.keras.Sequential(layers)


steps = 20*35000 // bs
opt = tf.optimizers.Adam()
loss_fn = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

model.build()

all_vars = [vars for vars in model.trainable_variables]

In [0]:
for seq_batch in data:
  for seq in seq_batch:
    print(seq)
    print("-----------------------------------")
    inp = tf.one_hot(seq, vocab_size, axis=-1)
    oh_seq = inp
    print(inp)

    y_pred = tf.roll(inp,-1,1)
    print("-----------------------------------")
    print(y_pred)
    y_pred_seq = tf.argmax(y_pred,1)
    print(y_pred_seq)
    break
  break

In [0]:
# @tf.function
def run_rnn_on_seq(seq_batch):
    model.reset_states()
    with tf.GradientTape() as tape:
        mask_len = [] #tf.TensorArray(tf.float32,size=128)
        for seq in seq_batch:
          mask_len.append(tf.math.count_nonzero(seq, dtype=tf.float32)-1,)

        mask_len = tf.stack(mask_len)
        mask = tf.sequence_mask(mask_len, seq_len, dtype=tf.float32)

        oh_seq  = tf.one_hot(seq_batch, vocab_size, axis=-1)
        y_actual = tf.roll(seq_batch, -1, 1)

        logits = model(oh_seq)
        losses = tf.nn.sparse_softmax_cross_entropy_with_logits(y_actual, logits)
        
        losses = losses * mask
        losses = tf.reduce_sum(losses, axis=1)

        
        xcent = tf.reduce_mean(tf.math.divide(losses,mask_len))

        # input()

    grads = tape.gradient(xcent, all_vars)
    
    # this is gradient clipping
    glob_norm = tf.linalg.global_norm(grads)
    grads = [g/glob_norm for g in grads]
    
    opt.apply_gradients(zip(grads, all_vars))

    return xcent

In [0]:
for step, seqs in enumerate(data):
    xent_avg = run_rnn_on_seq(seqs)

    if not step % 100:
        print("Step: {} Loss: {}".format(step, xent_avg))
        print()

    if step > steps:
        break

Step: 0 Loss: 1.522965669631958

Step: 100 Loss: 1.5471628904342651



KeyboardInterrupt: ignored

In [0]:
# create new model for generating text

layers_gen = [tf.keras.Input(shape=(1,vocab_size), batch_size=1),
          tf.keras.layers.SimpleRNN(n_h, return_sequences=True, stateful=True),
          tf.keras.layers.Dense(vocab_size)]

model_gen = tf.keras.Sequential(layers_gen)

model_gen.build()

model_gen.set_weights(model.get_weights())

model_gen.summary()


Model: "sequential_11"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
simple_rnn_11 (SimpleRNN)    (1, 1, 512)               298496    
_________________________________________________________________
dense_11 (Dense)             (1, 1, 70)                35910     
Total params: 334,406
Trainable params: 334,406
Non-trainable params: 0
_________________________________________________________________


In [0]:
def gen_seq(n_seq):        
  for _ in range(n_seq):
    char = 1      
    txt = []
    while not char == 2:
      gen = tf.Variable([[tf.one_hot(char, vocab_size)]])
      out = model_gen(gen)
      probs = tf.reshape(tf.nn.softmax(out),[-1]).numpy()
      char = np.random.choice(vocab_size, p=probs)      
      txt.append(char)
      if char == 2:
        model_gen.reset_states()
        break

    print("".join([ind_to_ch[ind] for ind in txt]).replace("</S>", "\n\n"))

        
gen_seq(50)


POMIEN:
O mence veait like to thee, geat tulick, matce.


BAUNIS:
Inco, awn al Eakera! hepofill Cimunice.


HENMI IEAM:
Yer, all ho's ancle;
That shillest on
to Bit good now.


JULIET:
I think's tor the.


CORNWALL:
If it our wanchowond.


PRoTHUS:
Ben how ar'I you.


ARIEL:
A deasm a moon.


ButhURCE:
I wmarrns voogevul?


RISCLIO:
Tour froment henrem!


BRATURSUCERO
In this coms?


GLOUCESTER:
Feas tein Prusees hope tise on my sun?


CELIA:
Say, your frain go?


CURI HENRO DO:
O, Rachorain so; foor houssthid with mes.


ORLANDO:
And thou dose that staly 'dese.


BENEDICK:
Come follow, betowe: whicherulow, is our neath.


MARK ANTONY:
I hay not oved in by dit.


WARLICK:
For Vellesie! How his winling of it io this leve nog.


LATY AAN
Prais, and elpare, and pon-alf,
Rpack, munds?


GLOUCESTER:
Come, will, Ay, Hirge: the bedre nciur,
I lak our thend.


ISaBELLA:


ANGELO:
Whot, apotiy rivedsely tliens are your sead a done.


GLOUCESTER:


Second Senacom:
Ot were in her to uplake, lite 